In [ ]:
import os
import re
import json
import copy

from dotenv import load_dotenv

import pandas as pd
from pathlib import Path

In [ ]:
# --- Caricamento RUNTS e Comuni + helper per DATI_STRUTTURATI (NO FUZZY) ---

COMUNI_CSV_PATH = os.path.join(f"classification/", f"cls_comuni.csv")

def _norm(s: str) -> str:
    return " ".join((s or "").strip().upper().split())

def load_comuni_validi(path: str) -> set:
    df = pd.read_csv(path, encoding="latin-1", dtype=str).fillna("")
    df.columns = [c.strip() for c in df.columns]
    # prova a trovare una colonna plausibile
    comune_col = None
    for c in df.columns:
        if c.lower() in ("comune", "nome_comune", "denominazione", "denominazione_comune"):
            comune_col = c
            break
    if not comune_col:
        # fallback: prima colonna
        comune_col = df.columns[0]
    comuni = set(_norm(x) for x in df[comune_col].tolist() if str(x).strip())
    if not comuni:
        raise RuntimeError(f"cls_comuni.csv vuoto o colonna comune non valida: {path}")
    return comuni




def extract_comuni_candidates(text: str) -> list:
    # Estrae solo candidati "espliciti" tipo "Comune di X"
    if not text:
        return []
    out = []
    for m in re.finditer(r"\bCOMUNE\s+DI\s+([A-ZÀ-Ù'\- ]{2,})", text, flags=re.IGNORECASE):
        out.append(m.group(1).strip())
    # dedup
    seen=set(); res=[]
    for x in out:
        k=_norm(x)
        if k not in seen:
            seen.add(k); res.append(x)
    return res

def validate_comuni(estratti: list, comuni_validi: set) -> list[dict]:
    validated = []
    for c in estratti:
        k = _norm(c)
        if k in comuni_validi:
            validated.append({"nome_estratto": c, "comune_validato": k, "valido": True})
        else:
            validated.append({"nome_estratto": c, "comune_validato": "", "valido": False})
    return validated

# Carica una volta sola (cache in memoria)
comuni_validi = load_comuni_validi(COMUNI_CSV_PATH)
print("✅ Comuni caricati:", len(comuni_validi))


✅ Comuni caricati: 7896
